In [1]:
import jax.random as random
import jax.numpy as jnp
from models import IQPTensorNetwork, local_gates
nqubits = 5
gate_list = local_gates(nqubits, max_weight=3)
iqp = IQPTensorNetwork(nqubits=nqubits, interactions=gate_list)

key = random.PRNGKey(42)
params = random.uniform(key, shape=(len(gate_list),), minval=0, maxval=2*jnp.pi)

circuit = iqp.build_circuit(parameters=params)

In [ ]:
from expectation import expvals_sampling, expvals_mc
from utils import convert_to_jnp_ndarray
from expectation import expvals_contraction

operators = [(0,1),(1,2),(3,4)]
expval_exact = expvals_contraction(circ=circuit, sites=operators)

operators_vec = convert_to_jnp_ndarray(operators, n_qubits=5)
generators = convert_to_jnp_ndarray(gate_list, n_qubits=5)

mean_sample, sigma_sample = expvals_sampling(circuit, operators, 100)
mean_mc, sigma_mc =  expvals_mc(
    params=params, 
    generators=generators, 
    ops=operators_vec, 
    n_samples=1000,
    key=key
    )

Check compatibility

In [13]:
def check():
    if expval_exact.shape != mean_sample.shape:
        raise ValueError(
            f"Forma non compatibile: expval_exact {expval_exact.shape}, "
            f"mean_sample {mean_sample.shape}"
        )

    # Verifica compatibilità tra le due stime: |m1 - m2| <= k * sqrt(s1^2 + s2^2)
    k = 2

    delta = jnp.abs(mean_sample - mean_mc)
    sigma_comb = jnp.sqrt(sigma_sample**2 + sigma_mc**2)
    z = delta / sigma_comb
    compatibili = z <= k

    # Compatibilità con il valore esatto (assumendo incertezza nulla sull'esatto)
    delta_sample_exact = jnp.abs(mean_sample - expval_exact)
    z_sample_exact = delta_sample_exact / sigma_sample
    compat_sample_exact = z_sample_exact <= k

    delta_mc_exact = jnp.abs(mean_mc - expval_exact)
    z_mc_exact = delta_mc_exact / sigma_mc
    compat_mc_exact = z_mc_exact <= k

    print("expval esatto:", expval_exact)

    print("\n[Sample vs MC]")
    print("delta:", delta)
    print("sigma combinata:", sigma_comb)
    print("z-score:", z)
    print(f"Compatibili entro {k}σ:", compatibili)
    print("Compatibilità globale:", bool(jnp.all(compatibili)))

    print("\n[Sample vs Esatto]")
    print("delta:", delta_sample_exact)
    print("z-score:", z_sample_exact)
    print(f"Compatibili entro {k}σ:", compat_sample_exact)
    print("Compatibilità globale:", bool(jnp.all(compat_sample_exact)))

    print("\n[MC vs Esatto]")
    print("delta:", delta_mc_exact)
    print("z-score:", z_mc_exact)
    print(f"Compatibili entro {k}σ:", compat_mc_exact)
    print("Compatibilità globale:", bool(jnp.all(compat_mc_exact)))
check()

expval esatto: [ 0.17861146  0.16566035 -0.28041425]

[Sample vs MC]
delta: [0.03631119 0.18467605 0.10868895]
sigma combinata: [0.10201387 0.10236832 0.1013366 ]
z-score: [0.3559437 1.8040351 1.0725538]
Compatibili entro 2σ: [ True  True  True]
Compatibilità globale: True

[Sample vs Esatto]
delta: [0.07861146 0.06566036 0.08041427]
z-score: [0.78611475 0.65660363 0.8166107 ]
Compatibili entro 2σ: [ True  True  True]
Compatibilità globale: True

[MC vs Esatto]
delta: [0.04230027 0.2503364  0.18910322]
z-score: [ 2.0971773 11.434854   7.9058733]
Compatibili entro 2σ: [False False False]
Compatibilità globale: False
